In [ ]:
from board import Loc, Target, Cell, Node, Board, transform, parse, DIGITS, POS9

## A puzzle


In [ ]:
puzzle = parse("""
.8.....52
.......87
....98...
4...3.6..
.2.7.....
.........
6..8.2...
...5.91..
9........
""")

## UI


In [ ]:
from typing import Iterable, Any
from ipywidgets import widgets as w
from ipycanvas import hold_canvas
from canvas import SudokuCanvas

# global puzzzle
anchors = None
links = None
chains = None

In [ ]:
from canvas import PALETTE_T10


debug_view = w.Output()

LINKSTYLES = {"HLink": "HARD", "SLink": "SOFT"}
column_layout = w.Layout(width="auto", height="100%", flex_flow="column", align_items="stretch")


canvas = SudokuCanvas()
# selecting_layers = w.SelectMultiple(
#     options=(0,) + DIGITS,
#     value=[0],
#     layout=column_layout,
# )
selecting_targets = w.SelectMultiple(
    options=[],
    value=[],
    layout=column_layout,
    style=dict(description_width="0"),
)
selecting_links = w.SelectMultiple(
    options=[],
    value=[],
    layout=column_layout,
)
selecting_chains = w.Select(
    options=[],
    value=None,
    layout=column_layout,
)


def deselect(widget):
    widget.value = () if isinstance(widget, w.SelectMultiple) else None


@canvas.on_client_ready
def init_canvas():
    canvas[2].global_alpha = 0.5
    canvas.draw_grid()


@canvas.on_mouse_up
def on_canvas_click(x, y):
    targ = canvas.map_target(x, y)

    if not any(t == targ for _, t in selecting_targets.options):
        return

    if targ in selecting_targets.value:
        if selecting_targets.value is not None:
            selecting_targets.value = [v for v in selecting_targets.value if v != targ]
    else:
        if selecting_targets.value is not None:
            selecting_targets.value += (targ,)
        else:
            selecting_targets.value = (targ,)


# @selecting_layers.observe
# def on_select_layer(change):
#     selected = change.new
#     canvas.clear_highlights()
#     if len(selected) == 0:
#         return
#     with hold_canvas():
#         for digit in selected:
#             for target in iter_layer(puzzle, digit):
#                 canvas.highlight_target(target)


@selecting_targets.observe
def on_select_target(change):
    if change.name != "value":
        return

    print("targets", change)

    if len(change.old):
        with hold_canvas():
            canvas.clear_highlights()

    selected = change.new
    if len(selected):
        deselect(selecting_links)
        deselect(selecting_chains)

        with hold_canvas():
            for target in selected:
                canvas.highlight_target(target)


@selecting_links.observe
def on_select_link(change):
    if change.name != "value":
        return

    if len(change.old):
        with hold_canvas():
            canvas.clear_highlights()

    selected = change.new
    if len(selected):
        deselect(selecting_targets)
        deselect(selecting_chains)
        with hold_canvas():
            for lnk in selected:
                canvas.highlight_link(lnk, style=LINKSTYLES[lnk.__class__.__name__])
            for lnk in selected:
                canvas.highlight_target(lnk[0])
                canvas.highlight_target(lnk[1])


@selecting_chains.observe
def on_select_chain(change):
    if change.name != "value":
        return

    if change.old is not None:
        with hold_canvas():
            canvas.clear_highlights()

    selected = change.new
    if selected is not None:
        deselect(selecting_targets)
        deselect(selecting_links)
        with hold_canvas():
            color = PALETTE_T10["green"] if selected.is_cyclic else PALETTE_T10["cyan"]
            for lnk in selected:
                canvas.highlight_link(lnk, style=LINKSTYLES[lnk.__class__.__name__], color=color)
            for trg in selected.anchors():
                canvas.highlight_target(trg, color=color)


def format_options(objects: Iterable[Any]):
    return tuple((str(obj), obj) for obj in objects)


btn1 = w.Button(description="reload")


@btn1.on_click
def on_reload(btn):
    canvas.clear_highlights()
    if puzzle is not None:
        canvas.draw_board(puzzle)
    if anchors is not None:
        selecting_targets.options = format_options(anchors)
    if links is not None:
        selecting_links.options = format_options(links)
    if chains is not None:
        selecting_chains.options = format_options(chains)


w.HBox(
    [
        w.VBox([btn1]),
        canvas,
        w.VBox([w.Label("Anchors"), selecting_targets]),
        w.VBox([w.Label("Links"), selecting_links]),
        w.VBox([w.Label("Chains"), selecting_chains]),
    ],
    layout=dict(justify_content="flex-start", align_items="stretch"),
)

In [ ]:
debug_view

## Analyzing


In [ ]:
from typing import Iterable, Self, NamedTuple
from collections.abc import Generator
from itertools import chain as iterchain
from collections import deque

In [ ]:
def iter_layer(board: Board, digit: int) -> Generator[Target]:
    for node in board:
        if digit in node.cell:
            yield Target(node.loc, digit)

In [ ]:
from types import EllipsisType


class Locality(NamedTuple):
    """Locality of block, row, col or cell"""

    blk: int | EllipsisType
    row: int | EllipsisType
    col: int | EllipsisType

    def __iter__(self) -> Generator[Loc]:
        """Generate all locations in the locality"""
        match (self.blk, self.row, self.col):  # NB: cannot use self-match because of recursion
            case (int(b), EllipsisType(), EllipsisType()):
                b0 = b - 1
                r1 = 1 + 3 * (b0 // 3)
                c1 = 1 + 3 * (b0 % 3)
                yield from (Loc(r, c) for r in range(r1, r1 + 3) for c in range(c1, c1 + 3))
            case (EllipsisType(), int(r), int(c)):
                yield Loc(r, c)
            case (EllipsisType(), int(r), EllipsisType()):
                yield from (Loc(r, i) for i in POS9)
            case (EllipsisType(), EllipsisType(), int(c)):
                yield from (Loc(i, c) for i in POS9)
            case _:
                raise TypeError()

    def __contains__(self, loc: Loc) -> bool:
        """Chack if location is in the locality"""
        match (self.blk, self.row, self.col):
            case (int(b), EllipsisType(), EllipsisType()):
                return loc.blk == b
            case (EllipsisType(), int(r), int(c)):
                return loc.row == r and loc.col == c
            case (EllipsisType(), int(r), EllipsisType()):
                return loc.row == r
            case (EllipsisType(), EllipsisType(), int(c)):
                return loc.col == c
            case _:
                raise TypeError()

    @classmethod
    def common(cls, l1: Loc, l2: Loc) -> Generator[Self]:
        """Get all localities common for the locs"""
        if l1 == l2:
            yield cls(..., l1.row, l2.col)
            return
        if l1.blk == l2.blk:
            yield cls(l1.blk, ..., ...)
        if l1.row == l2.row:
            yield cls(..., l1.row, ...)
        if l1.col == l2.col:
            yield cls(..., ..., l1.row)

### Basic


In [ ]:
def fillempty(board: Board, node: Node):
    if node.cell.is_empty:
        return Node(node.loc, Cell(DIGITS))
    else:
        return node

In [ ]:
def cleanup(board: Board, node: Node) -> Node:
    """Clean up direct contradictions in localities"""

    def finals(loc: Locality):
        cells = [n.cell for n in board.slice(iter(loc))]
        return set(c.final for c in cells if c.is_final)

    if node.cell.is_final:
        return node
    blkfinals = finals(Locality(node.loc.blk, ..., ...))
    rowfinals = finals(Locality(..., node.loc.row, ...))
    colfinals = finals(Locality(..., ..., node.loc.col))
    allfinals = blkfinals | rowfinals | colfinals
    return Node(node.loc, Cell(set(node.cell) - allfinals))


In [ ]:
puzzle = transform(puzzle, fillempty)
puzzle = transform(puzzle, cleanup)

## Links


In [ ]:
class Link(tuple[Target, Target]):
    """Ordered set of targets
    (with symmetric equality)
    """

    def __str__(self):
        return f"{self[0]} ~ {self[1]}"

    def strtail(self):
        return f" ~ {self[1]}"

    def __repr__(self):
        return f"{self.__class__.__name__}(({self[0]!r}, {self[1]!r},))"

    def reversed(self):
        return self.__class__((self[1], self[0]))

    def __hash__(self):
        # symmetric hash
        return tuple.__hash__(self) + tuple.__hash__(self.reversed())

    def __eq__(self, other):
        # symmetric equality
        return hash(self) == hash(other)


class HLink(Link):
    """Hard link, XOR relation"""

    def __str__(self):
        return f"{self[0]}⟺{self[1]}"

    def strtail(self):
        return f"⟺{self[1]}"


class SLink(Link):
    """Soft link, NAND relation"""

    def __str__(self):
        return f"{self[0]}⟷{self[1]}"

    def strtail(self):
        return f"⟷{self[1]}"

### hard links

Represent XOR relation

Criteria (for signular targets):

- only 2 drafts of same digit in a locality
- only 2 drafts in a cell


In [ ]:
def find_hardlinks(board: Board) -> Generator[HLink]:

    def scan_cell(loc: Loc):
        node = board.get(loc)
        if len(node.cell) == 2:
            d1, d2 = node.cell
            yield HLink((
                Target(node.loc, d1),
                Target(node.loc, d2),
            ))

    def scan_locality(loc: Locality):
        nodes = board.slice(iter(loc))
        for d in DIGITS:
            sublayer = tuple(n for n in nodes if d in n.cell)
            if len(sublayer) == 2:
                n1, n2 = sublayer
                yield HLink((
                    Target(n1.loc, d),
                    Target(n2.loc, d),
                ))

    for node in board:
        yield from scan_cell(node.loc)
    for i in POS9:
        yield from scan_locality(Locality(i, ..., ...))
        yield from scan_locality(Locality(..., i, ...))
        yield from scan_locality(Locality(..., ..., i))


In [ ]:
links = set[HLink](find_hardlinks(puzzle))
anchors = set(iterchain.from_iterable(links))

In [ ]:
for lnk in links:
    print(lnk)

### soft links

Represent NAND relation

Criteria:

- any 2 drafts of same digit in a locality
- any 2 drafts in a cell


In [ ]:
def check_nand(t1: Target, t2: Target):
    l1 = t1.loc
    l2 = t2.loc
    if t1.seg == t2.seg:
        return l1.blk == l2.blk or l1.row == l2.row or l1.col == l2.col
    else:
        return l1 == l2

### chains

Alterating link chains: (-xor-nand-)^n

(A-xor-B-nand-)^n-xor-D => All (X nand A) and (X nand D) can be eliminated


In [ ]:
import re


class Chain(tuple[Link, ...]):
    @classmethod
    def init(cls, link: Link):
        return cls((link,))

    def __str__(self):
        return "".join([str(self[0])] + [lnk.strtail() for lnk in self[1:]])

    def __add__(self, other: Self):
        assert self[-1][-1] == other[0][0]
        return Chain(tuple(self) + tuple(other))

    def anchors(self) -> Iterable[Target]:
        """All anchor points in the chain"""
        return (self[0][0], *(lnk[1] for lnk in self))

    def ends(self):
        return (self[0][0], self[-1][1])

    def __hash__(self):
        """Hashing by unordered links"""
        return hash(frozenset(self))

    def __eq__(self, other: Self):
        return hash(self) == hash(other)

    @property
    def is_cyclic(self):
        e1, e2 = self.ends()
        return e1 == e2

    def pattern(self):
        """Returns string pattern of link classes like `HLink~SLink~`"""
        kinds = [lnk.__class__.__name__ for lnk in self]
        pattern = "~".join(kinds)
        if self.is_cyclic:
            return f"~{pattern}~"
        else:
            return pattern

In [ ]:
#### depth-first search for all ALC, starting from hard links

RE_ALC = re.compile(r"^(HLink~SLink~)+HLink$")
RE_ALCC = re.compile(r"^~(HLink~SLink~)+$")


def check_goal(chain: Chain):
    pattern = chain.pattern()
    return RE_ALCC.match(pattern) or RE_ALC.match(pattern)
    # return RE_ALCC.match(pattern)


def check_expansion(last: HLink, other: HLink) -> tuple[SLink, HLink] | None:
    front = last[1]
    if front in other:
        return None
    if check_nand(front, other[0]):
        return SLink((front, other[0])), other
    if check_nand(front, other[1]):
        return SLink((front, other[1])), other.reversed()


def expand_alc(links: set[HLink], chain: Chain) -> Generator[Chain]:
    last: HLink = chain[-1]  # type: ignore

    e1, e2 = chain.ends()

    if e1 != e2 and check_nand(e1, e2):
        yield chain + Chain((SLink((e2, e1)),))

    for other in links:
        if other not in chain:
            expansion = check_expansion(last, other)
            if expansion is not None and expansion[0] not in chain:
                yield chain + Chain(expansion)


def search_chains(links: set[HLink], initial: Link | None = None):
    if initial is not None:
        frontier = deque[Chain]((Chain.init(initial),))
    else:
        frontier = deque[Chain](Chain.init(lnk) for lnk in links)
    explored = set[Chain]()
    while frontier:
        chain = frontier.pop()
        if check_goal(chain):
            yield chain
        explored.add(chain)
        frontier.extend(ext for ext in expand_alc(links, chain) if ext not in explored and ext not in frontier)

In [ ]:
chains = set(search_chains(links))

In [ ]:
for ch in chains:
    print(ch)